# 🔬 Lab 06B: Multi-Format Ingestion & Interactive Knowledge Graph Engineering

---

## 🎯 Executive Overview & Architectural Challenge
In complex enterprise environments like **UST Global (UST)**, operational intelligence is deeply fragmented across disparate file formats:
1. **Unstructured Technical Text & SOPs (`.md`, `.txt`, `.pdf`)**: Operating envelopes, startup checklists, safety trip limits.
2. **Semi-Structured Maintenance Logs (`.json`)**: Equipment failure records, downtime telemetry, technician work orders.
3. **Structured Tabular Data (`.csv`)**: Sensor stream summaries, operating pressure ratios, vibration metrics.
4. **Engineering Source Code & Automation Scripts (`.py`, `.sql`)**: SCADA diagnostic scripts, predictive maintenance triggers, Cypher procedures.
5. **Multi-Modal Visual Assets (`.png`, `.jpg`)**: Compressor performance maps, P&ID instrumentation loops, surge lines.

### 🔴 Why Vector RAG Fails on Heterogeneous Silos:
Pure vector search flattens these diverse file formats into isolated text chunks. When an asset manager asks:
> *"Show me all critical seal failures at Fahud Compression Station, the root causes identified, which automation scripts monitor that equipment, and which lead engineers repaired them."*

Vector similarity search returns 3 disconnected chunks and completely misses the causal cross-document links!

### 🟢 The Knowledge Graph Solution (Inspired by Understand-Anything & GraphRAG):
We build an **Unified Multi-Format Ingestion & Knowledge Graph Pipeline** that parses 5+ diverse enterprise formats into a cohesive, interactive property graph:
- **Entities**: `Facility`, `Equipment`, `Subsystem`, `Incident`, `Engineer`, `Script`, `Sensor`, `Document`, `VisualDiagram`
- **Edges**: `OPERATES`, `HAS_SUBSYSTEM`, `EXPERIENCED_FAILURE`, `RESOLVED_BY`, `MONITORS_EQUIPMENT`, `AUTOMATES_MONITORING`, `AUTHORED`, `HAS_PERFORMANCE_CURVE`
- **Interactive Visualization**: Generates an interactive, physics-enabled HTML network graph (`pyvis`) with custom color-coded domains, physics simulation, node clustering, and dynamic neighborhood inspection.


In [ ]:
# Pre-flight imports & resilient environment setup
import os
import sys
import json
import csv
import re
from pathlib import Path
import networkx as nx
import matplotlib.pyplot as plt

# Ensure UTF-8 output
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

# Self-healing sys.path resolution so rag_universal_client is ALWAYS found
for candidate in [
    Path.cwd(),
    Path.cwd() / "RAG_ContxtEngg_Labs",
    Path(__file__).parent if "__file__" in locals() else None,
    Path.cwd() / "UST" / "RAG_ContxtEngg_Labs",
    Path(r"c:\Users\jadid\Desktop\MyWork\Techademy\UST\RAG_ContxtEngg_Labs")
]:
    if candidate and candidate.exists() and (candidate / "rag_universal_client.py").exists():
        if str(candidate.resolve()) not in sys.path:
            sys.path.insert(0, str(candidate.resolve()))
        break
import rag_universal_client as rc

# Resilient path resolution for sample_documents
DATA_DIR = rc.get_rag_data_dir()
print(f"[*] Verified Data Directory: {DATA_DIR}")

for f in DATA_DIR.iterdir():
    print(f"  - Found: {f.name} ({f.stat().st_size} bytes)")


---
## Step 1: Synthesizing Multi-Format Enterprise Knowledge Sources
To demonstrate cross-format ingestion, we prepare 5 complementary enterprise sources representing a real-world facility subsystem:
1. `turbomachinery_manual.md`: Technical O&M SOP and trip specifications.
2. `incident_reports.json`: Maximo/SAP maintenance incident records.
3. `scada_sensors.csv`: Sensor telemetry streams with operating bands.
4. `telemetry_monitor.py`: Automation script monitoring compressor health.
5. `compressor_curve.png`: Turbomachinery performance map.


In [ ]:
# Synthesize complementary multi-format operational data sources
# 1. SCADA Sensor CSV
scada_csv_path = DATA_DIR / "scada_telemetry.csv"
scada_records = [
    {"sensor_id": "PT-301-A", "equipment_tag": "K-301A", "metric": "Discharge Pressure", "unit": "bar", "nominal": "48.5", "alarm_high": "58.0"},
    {"sensor_id": "TT-301-B", "equipment_tag": "K-301A", "metric": "Bearing Temperature", "unit": "degC", "nominal": "72.0", "alarm_high": "95.0"},
    {"sensor_id": "VT-301-X", "equipment_tag": "K-301A", "metric": "Vibration Velocity", "unit": "mm/s", "nominal": "2.4", "alarm_high": "4.5"},
    {"sensor_id": "FT-102-S", "equipment_tag": "P-102B", "metric": "Flow Rate", "unit": "m3/hr", "nominal": "120.0", "alarm_high": "150.0"}
]
with open(scada_csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=scada_records[0].keys())
    writer.writeheader()
    writer.writerows(scada_records)

# 2. Python Automation Script (.py)
script_path = DATA_DIR / "telemetry_monitor.py"
sample_script_code = '''"""
UST SCADA Automated Monitoring Service
Author: Salim Al-Harthy (Turbomachinery Lead)
Monitors: K-301A Centrifugal Compressor at Fahud Central Compression
"""
import time

def monitor_compressor_vibration(equipment_tag="K-301A", threshold=4.5):
    # Connects to OPC-UA server for Fahud Compression Station
    current_vibration = 4.8  # Alert condition
    if current_vibration > threshold:
        trigger_alarm(equipment_tag, "High Vibration Warning on Tandem Dry Gas Seals")

def trigger_alarm(tag, reason):
    print(f"[ALARM] {tag}: {reason}")
'''
with open(script_path, "w", encoding="utf-8") as f:
    f.write(sample_script_code)

print(f"[+] Created synthetic CSV: {scada_csv_path.name}")
print(f"[+] Created synthetic Python script: {script_path.name}")


---
## Step 2: Multi-Format Parsers & Domain Extraction
Now we build specialized parser extractors for each file type:
- **Markdown Parser**: Extracts Equipment operating envelopes, safety limits, and document sections.
- **JSON Parser**: Extracts Incident events, severity ratings, downtime hours, root causes, and assigned engineers.
- **CSV Parser**: Extracts Sensor IDs, monitored telemetry metrics, units, and safety thresholds.
- **Code Parser (Python AST & Docstring)**: Extracts Script names, author attribution, target monitored equipment, and function routines.
- **Image/Vision Parser**: Extracts visual diagrams, performance curves, and equipment associations.


In [ ]:
# Define multi-format knowledge extraction pipeline
class MultiFormatKGIngestor:
    def __init__(self):
        self.triples = []
        self.nodes = {}
        
    def add_node(self, node_id, label, category, properties=None):
        if node_id not in self.nodes:
            self.nodes[node_id] = {
                "id": node_id,
                "label": label,
                "category": category,
                "properties": properties or {}
            }
        else:
            # Update properties if already exists
            if properties:
                self.nodes[node_id]["properties"].update(properties)
            
    def add_edge(self, source, target, relation, properties=None):
        # Guarantee both endpoints exist in nodes registry
        if source not in self.nodes:
            self.nodes[source] = {"id": source, "label": source, "category": "Entity", "properties": {}}
        if target not in self.nodes:
            self.nodes[target] = {"id": target, "label": target, "category": "Entity", "properties": {}}
            
        self.triples.append({
            "source": source,
            "target": target,
            "relation": relation,
            "properties": properties or {}
        })

    def parse_markdown_manual(self, filepath):
        with open(filepath, "r", encoding="utf-8") as f:
            text = f.read()
        
        # Extract Equipment & Safety Limits
        self.add_node("DOC_MANUAL", "Turbomachinery O&M Manual", "Document", {"format": "markdown", "path": filepath.name})
        
        # Regex extraction for turbine and compressor sections
        if "Solar Taurus 60" in text or "Centrifugal Compressor" in text:
            self.add_node("EQ_K301A", "K-301A Compressor", "Equipment", {"model": "Solar Taurus 60", "type": "Centrifugal Compressor"})
            self.add_edge("DOC_MANUAL", "EQ_K301A", "SPECIFIES_PROCEDURE")
            
        if "Tandem dry gas seal" in text:
            self.add_node("SUB_SEAL", "Tandem Dry Gas Seals", "Subsystem", {"limit_leakage": "45 Nm3/hr"})
            self.add_edge("EQ_K301A", "SUB_SEAL", "HAS_SUBSYSTEM")
            
        if "Level 2 ESD trip" in text:
            self.add_node("LIMIT_ESD2", "Level 2 ESD Trip (15,650 RPM)", "SafetyLimit", {"trip_speed": "15,650 RPM"})
            self.add_edge("EQ_K301A", "LIMIT_ESD2", "GOVERNED_BY")

    def parse_incident_json(self, filepath):
        with open(filepath, "r", encoding="utf-8") as f:
            incidents = json.load(f)
            
        for inc in incidents:
            inc_id = f"INC_{inc['incident_id']}"
            fac_id = f"FAC_{inc['facility'].split()[0].upper()}"
            eq_id = f"EQ_{inc['equipment_tag'].replace('-', '')}"
            eng_id = f"ENG_{inc['lead_engineer'].replace(' ', '_').upper()}"
            
            # Nodes
            self.add_node(fac_id, inc["facility"], "Facility", {"location": "UST Infrastructure Facility"})
            self.add_node(eq_id, f"Tag {inc['equipment_tag']}", "Equipment", {"tag": inc['equipment_tag']})
            self.add_node(inc_id, f"Incident {inc['incident_id']}", "Incident", {
                "severity": inc["severity"],
                "downtime_hours": inc["downtime_hours"],
                "root_cause": inc["root_cause"]
            })
            self.add_node(eng_id, inc["lead_engineer"], "Engineer", {"role": "Lead Maintenance Engineer"})
            
            # Edges
            self.add_edge(fac_id, eq_id, "OPERATES")
            self.add_edge(eq_id, inc_id, "EXPERIENCED_FAILURE")
            self.add_edge(inc_id, eng_id, "RESOLVED_BY")
            
            # Subsystem edge
            sub_id = f"SUB_{inc['subsystem'][:10].replace(' ', '_').upper()}"
            self.add_node(sub_id, inc["subsystem"], "Subsystem")
            self.add_edge(eq_id, sub_id, "HAS_SUBSYSTEM")

    def parse_scada_csv(self, filepath):
        with open(filepath, "r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                sensor_id = f"SENSOR_{row['sensor_id'].replace('-', '_')}"
                eq_id = f"EQ_{row['equipment_tag'].replace('-', '')}"
                
                # Ensure equipment node exists with proper label
                self.add_node(eq_id, f"Tag {row['equipment_tag']}", "Equipment", {"tag": row['equipment_tag']})
                self.add_node(sensor_id, f"{row['sensor_id']} ({row['metric']})", "Sensor", {
                    "metric": row["metric"],
                    "unit": row["unit"],
                    "nominal": row["nominal"],
                    "alarm_high": row["alarm_high"]
                })
                self.add_edge(sensor_id, eq_id, "MONITORS_EQUIPMENT")

    def parse_python_script(self, filepath):
        with open(filepath, "r", encoding="utf-8") as f:
            code = f.read()
            
        script_node_id = f"SCRIPT_{filepath.stem.upper()}"
        self.add_node(script_node_id, filepath.name, "Script", {"language": "python", "path": filepath.name})
        
        # Link to author
        if "Salim Al-Harthy" in code:
            self.add_edge("ENG_SALIM_AL-HARTHY", script_node_id, "AUTHORED")
            
        # Link to target equipment
        if "K-301A" in code:
            self.add_edge(script_node_id, "EQ_K301A", "AUTOMATES_MONITORING")

    def parse_image_asset(self, filepath):
        img_node_id = f"IMG_{filepath.stem.upper()}"
        self.add_node(img_node_id, filepath.name, "VisualDiagram", {"format": filepath.suffix, "type": "Compressor Performance Map"})
        self.add_edge("EQ_K301A", img_node_id, "HAS_PERFORMANCE_CURVE")

# Execute multi-format ingestion
ingestor = MultiFormatKGIngestor()
ingestor.parse_markdown_manual(DATA_DIR / "turbomachinery_manual.md")
ingestor.parse_incident_json(DATA_DIR / "incident_reports.json")
ingestor.parse_scada_csv(scada_csv_path)
ingestor.parse_python_script(script_path)
ingestor.parse_image_asset(DATA_DIR / "compressor_curve.png")

print(f"[+] Multi-Format Ingestion Complete:")
print(f"  - Extracted Nodes: {len(ingestor.nodes)}")
print(f"  - Extracted Directed Edges: {len(ingestor.triples)}")


---
## Step 3: Graph Assembly & Domain Ontology Categorization
We assemble the extracted entities into a NetworkX directed property graph `G`. Each node belongs to a distinct **Architectural Domain** (similar to the business domain breakdown in *Understand-Anything*):
- 🟢 **Facilities & Locations**: Physical operational concession plants.
- 🔵 **Equipment & Subsystems**: Critical machinery and components.
- 🔴 **Incidents & Safety**: Operational failures, root causes, and trip thresholds.
- 🟣 **Personnel & Authors**: Maintenance engineers and script creators.
- 🟡 **Automation & Telemetry**: SCADA sensors, scripts, and visual performance curves.


In [ ]:
# Build NetworkX Multi-Domain Graph
G = nx.DiGraph()

# Color mapping by domain category
CATEGORY_COLORS = {
    "Facility": "#10b981",       # Emerald
    "Equipment": "#3b82f6",      # Blue
    "Subsystem": "#06b6d4",      # Cyan
    "Incident": "#f43f5e",       # Rose
    "SafetyLimit": "#fbbf24",    # Amber
    "Engineer": "#a855f7",       # Purple
    "Sensor": "#eab308",         # Yellow
    "Script": "#6366f1",         # Indigo
    "Document": "#64748b",       # Slate
    "VisualDiagram": "#ec4899"   # Pink
}

for node_id, data in ingestor.nodes.items():
    G.add_node(
        node_id,
        label=data["label"],
        category=data["category"],
        color=CATEGORY_COLORS.get(data["category"], "#94a3b8"),
        **data["properties"]
    )

for edge in ingestor.triples:
    G.add_edge(
        edge["source"],
        edge["target"],
        relation=edge["relation"],
        **edge["properties"]
    )

print(f"[+] Graph Topology Compiled:")
print(f"  - Nodes: {G.number_of_nodes()}")
print(f"  - Edges: {G.number_of_edges()}")
print(f"  - Connected Components: {nx.number_weakly_connected_components(G)}")


---
## Step 4: Static Topology Visualization (Matplotlib)
Let's first render a high-contrast publication-grade static graph layout using NetworkX spring layout.


In [ ]:
plt.figure(figsize=(14, 9), dpi=140, facecolor='#070a13')
pos = nx.spring_layout(G, k=1.2, seed=42)

# Collect node colors from attributes
node_colors = [G.nodes[n].get('color', '#38bdf8') for n in G.nodes()]

# Draw network nodes with subtle borders
nx.draw_networkx_nodes(G, pos, node_size=2000, node_color=node_colors, alpha=0.9, edgecolors='#ffffff', linewidths=1.2)
nx.draw_networkx_labels(G, pos, labels={n: G.nodes[n].get('label', n) for n in G.nodes()}, font_size=7, font_color='#ffffff', font_weight='bold')
nx.draw_networkx_edges(G, pos, edge_color='#475569', arrowsize=14, width=1.4, alpha=0.7, connectionstyle='arc3,rad=0.08')

# Edge labels
edge_labels = {(u, v): d['relation'] for u, v, d in G.edges(data=True)}
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=6, font_color='#fbbf24', font_family='sans-serif')

plt.title('UST Enterprise Operations Multi-Format Knowledge Graph', fontsize=13, fontweight='bold', color='#ffffff', pad=15)
plt.axis('off')
plt.tight_layout()
plt.show()


---
## Step 5: Interactive Visual Knowledge Graph (PyVis / Vis.js)
Following the architecture of interactive tools like **Understand-Anything**, we export our multi-format knowledge graph into an interactive, physics-driven HTML visualization with:
- **Hoverable Tooltips**: Displays full properties (nominal sensor values, downtime hours, script authors).
- **Physics Simulation (Barnes-Hut)**: Drag nodes, inspect clustering, and pull sub-graphs apart.
- **Dynamic Neighborhood Highlight**: Clicking an equipment node illuminates all linked sensors, incidents, scripts, and engineers!


In [ ]:
from pyvis.network import Network
from IPython.display import display, HTML

# Initialize interactive PyVis Network with dark theme styling
net = Network(height="650px", width="100%", bgcolor="#070a13", font_color="#f8fafc", directed=True)

# Configure physics solver (Barnes-Hut layout)
net.barnes_hut(gravity=-3000, central_gravity=0.3, spring_length=160, spring_strength=0.05, damping=0.09)

# Add nodes with rich HTML tooltips
for node_id, data in G.nodes(data=True):
    category = data.get("category", "General")
    color = data.get("color", "#38bdf8")
    label = data.get("label", node_id)
    
    # Build detailed tooltip
    tooltip_lines = [f"<b>{label}</b>", f"Category: <span style='color:{color}'>{category}</span>", "<hr style='margin:4px 0; border:0.5px solid #475569'>"]
    for k, v in data.items():
        if k not in ["label", "category", "color"]:
            tooltip_lines.append(f"• <b>{k}</b>: {v}")
    title_html = "<div style='font-family:Inter,sans-serif; font-size:12px; line-height:1.4;'>" + "<br>".join(tooltip_lines) + "</div>"
    
    net.add_node(
        node_id,
        label=label,
        title=title_html,
        color=color,
        size=25 if category in ["Facility", "Equipment"] else 18,
        borderWidth=2,
        shadow=True
    )

# Add edges with relationship labels
for u, v, data in G.edges(data=True):
    rel = data.get("relation", "LINKED_TO")
    net.add_edge(u, v, label=rel, color="#64748b", arrows="to", font={"size": 9, "color": "#fbbf24", "align": "middle"})

# Export to standalone HTML file
output_html = Path("interactive_multiformat_knowledge_graph.html")
net.save_graph(str(output_html))
print(f"[+] Interactive Knowledge Graph generated: {output_html.resolve()}")

# Display iframe directly in Jupyter
display(HTML(f'''
<div style="border: 2px solid #3b82f6; border-radius: 12px; overflow: hidden; margin-top: 10px;">
    <iframe src="{output_html.name}" width="100%" height="670px" style="border:none;"></iframe>
</div>
'''))


---
## Step 6: Multi-Hop Cross-Format Graph Traversal & Diagnostic Reasoning
Now let's verify how this knowledge graph solves complex multi-hop queries that pure vector search fails to answer.

**Enterprise Diagnostic Query:**
> *"Investigate equipment K-301A: What facility operates it, what failure did it experience, who resolved it, what sensors monitor it, and what automation script oversees its operation?"*


In [ ]:
def execute_cross_format_investigation(equipment_id, graph):
    eq_label = graph.nodes[equipment_id].get('label', equipment_id)
    print(f"=== Multi-Hop Diagnostic Report for: {eq_label} ===\n")
    
    # 1. Inward Edges (Facilities operating it, Sensors monitoring it, Scripts automating it)
    in_edges = graph.in_edges(equipment_id, data=True)
    facilities = []
    sensors = []
    scripts = []
    docs = []
    
    for u, v, d in in_edges:
        u_data = graph.nodes[u]
        cat = u_data.get("category")
        rel = d.get("relation")
        if cat == "Facility":
            facilities.append(f"{u_data['label']} ({u})")
        elif cat == "Sensor":
            sensors.append(f"{u_data['label']} [Nominal: {u_data.get('nominal')} {u_data.get('unit')}, Alarm: {u_data.get('alarm_high')}]")
        elif cat == "Script":
            scripts.append(f"{u_data['label']} (Path: {u_data.get('path')})")
        elif cat == "Document":
            docs.append(u_data['label'])
            
    print(f"Operational Facility: {', '.join(facilities) or 'Unknown'}")
    print(f"Governing SOP: {', '.join(docs) or 'Standard SOP'}")
    print(f"Monitored SCADA Sensors ({len(sensors)} sensors):")
    for s in sensors:
        print(f"   - {s}")
    print(f"Automation Script: {', '.join(scripts) or 'None'}")
    
    # 2. Outward Edges (Subsystems, Incidents, Visual Curves)
    out_edges = graph.out_edges(equipment_id, data=True)
    for u, v, d in out_edges:
        v_data = graph.nodes[v]
        cat = v_data.get("category")
        rel = d.get("relation")
        
        if cat == "Subsystem":
            print(f"Subsystem: {v_data['label']} (Rel: {rel})")
        elif cat == "VisualDiagram":
            print(f"Visual Curve Asset: {v_data['label']} (Type: {v_data.get('type')})")
        elif cat == "Incident":
            print(f"Recorded Incident: {v_data['label']}")
            print(f"   - Severity: {v_data.get('severity')} | Downtime: {v_data.get('downtime_hours')} hrs")
            print(f"   - Root Cause: {v_data.get('root_cause')}")
            
            # 2nd Hop: Assigned Engineer
            for inc_u, eng_v, inc_d in graph.out_edges(v, data=True):
                if graph.nodes[eng_v].get("category") == "Engineer":
                    eng = graph.nodes[eng_v]
                    print(f"   - Assigned Resolution Engineer: {eng['label']} (ID: {eng_v})")

# Execute investigation
execute_cross_format_investigation("EQ_K301A", G)


---
## Step 7: Exporting to Enterprise Neo4j Cypher Schema
To operationalize this multi-format graph in enterprise production (e.g. on Neo4j Community/Enterprise), we export declarative parameterized Cypher statements.


In [ ]:
def export_to_production_cypher(graph):
    statements = ["// ==============================================", "// UST MULTI-FORMAT KNOWLEDGE GRAPH CYPHER DDL", "// =============================================="]
    
    # 1. Create Nodes
    for n, d in graph.nodes(data=True):
        cat = d.get("category", "Entity")
        props = {k: v for k, v in d.items() if k not in ["color"]}
        props_str = json.dumps(props)
        # Convert JSON string to Cypher map format
        props_cypher = re.sub(r'"([a-zA-Z_]+)":', r'\1:', props_str)
        statements.append(f"MERGE (:{cat} {{id: '{n}'}}) ON CREATE SET n = {props_cypher};")
        
    # 2. Create Directed Edges
    for u, v, d in graph.edges(data=True):
        rel = d.get("relation", "LINKED_TO")
        statements.append(f"MATCH (a {{id: '{u}'}}), (b {{id: '{v}'}}) MERGE (a)-[:{rel}]->(b);")
        
    return "\n".join(statements)

cypher_script = export_to_production_cypher(G)
cypher_file = Path("ust_multiformat_graph_schema.cypher")
cypher_file.write_text(cypher_script, encoding="utf-8")

print(f"[+] Exported {len(cypher_script.splitlines())} Cypher statements to {cypher_file.name}")
print("\n--- Sample Cypher Statements (First 10 Lines) ---")
for line in cypher_script.splitlines()[:12]:
    print(line)


---
## 🏁 Production Engineering Takeaways & Enterprise Best Practices
1. **Multi-Format Extraction Over Single-File RAG**: In industrial domains, no single document holds the full truth. Ingesting code, telemetry CSVs, PDF manuals, and Maximo JSON together into a graph unlocks cross-silo causality.
2. **Interactive Visual Graphs for Human-in-the-Loop AI**: Tools like `pyvis` and *Understand-Anything* transform impenetrable graph databases into intuitive visual dashboards where domain experts can validate entity connections before deployment.
3. **Hybrid Vector + Graph Retrieval**: Vector search identifies the target entry nodes ($N_0$), while the Knowledge Graph expands outward to retrieve all connected telemetry, code, and incident history for grounded LLM synthesis.
